In [1]:
import os
import sys
import glob
import warnings
import time
import numpy as np
from scipy import stats, signal
from scipy.io import loadmat
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score
)
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import logging
import joblib
import cv2

warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [2]:
class Config:
    """Central configuration for the pipeline."""

    # Dataset path (update this to your UTD-MHAD location)
    DATASET_ROOT = './UTD-MHAD Dataset'

    # Subdirectories within UTD-MHAD
    SKELETON_DIR = "Skeleton"
    INERTIAL_DIR = "Inertial"
    DEPTH_DIR = "Depth"
    VIDEO_DIRS = ["RGB-part1", "RGB-part2", "RGB-part3", "RGB-part4"]

    # Output directory for saved features (relative to notebook location)
    FEATURES_DIR = "features_loss_50"

    # Dataset properties
    NUM_ACTIONS = 27
    NUM_SUBJECTS = 8
    NUM_TRIALS = 4
    NUM_JOINTS = 20  # Kinect v1 skeleton has 20 joints

    # Train/test split: odd subjects for training, even for testing
    # (standard protocol for UTD-MHAD)
    TRAIN_SUBJECTS = [1, 3, 5, 7]
    TEST_SUBJECTS = [2, 4, 6, 8]

    # Inertial feature extraction parameters (inspired by VISTA paper)
    INERTIAL_WINDOW_SIZE = 50     # samples per window
    INERTIAL_WINDOW_OVERLAP = 0.5  # 50% overlap

    # Depth feature extraction parameters
    DEPTH_SAMPLE_FRAMES = 10  # number of frames to sample from each sequence

    # Video (RGB) feature extraction parameters (inspired by VISTA paper)
    VIDEO_SAMPLE_FRAMES = 15   # number of frames to uniformly sample
    VIDEO_RESIZE = (120, 160)  # (height, width) for downsampled frames

    # Random Forest parameters
    RF_N_ESTIMATORS = 300
    RF_MAX_DEPTH = None
    RF_MIN_SAMPLES_SPLIT = 2
    RF_MIN_SAMPLES_LEAF = 1
    RF_RANDOM_STATE = 42
    RF_N_JOBS = -1

    # Whether to use each modality
    USE_SKELETON = True
    USE_INERTIAL = True
    USE_DEPTH = True
    USE_VIDEO = True

    # Simulated raw data loss parameters
    # Applied BEFORE feature extraction on the raw structural units
    LOSS_RATE = 0.50        # 20% of structural units are zeroed out
    LOSS_SEED = 42          # Random seed for reproducibility


In [3]:
# =============================================================================
# SECTION 2: DATA LOADING  –  lazy / filename-only registry
# =============================================================================
#
# UTDMHADLoader now works in two stages:
#
#   Stage 1 – build_sample_registry()
#       Scans all four modality directories and records ONLY file paths.
#       No raw data is read into memory.  Every sample entry holds just
#       metadata (action, subject, trial) and the path strings.
#
#   Stage 2 – on-demand per-file loaders  (called once per sample)
#       load_skeleton_from_file(path)  -> (frames, 20, 3) float64 | None
#       load_inertial_from_file(path)  -> (samples, 6) float64    | None
#       load_depth_from_file(path)     -> (H, W, frames) float64  | None
#       (video paths are passed directly to VideoFeatureExtractor)
#
# Peak memory = one sample's raw arrays + all accumulated feature vectors.
# =============================================================================

class UTDMHADLoader:
    """
    Lazy loader for the UTD-MHAD dataset.

    build_sample_registry() returns a dict::

        { sample_key: {
            'action': int, 'subject': int, 'trial': int,
            'skeleton_path': str | None,
            'inertial_path': str | None,
            'depth_path':    str | None,
            'video_path':    str | None,
          }
        }

    Raw data is NEVER stored in this object.  Callers load one sample
    at a time via the static load_*_from_file() helpers.
    """

    def __init__(self, config):
        self.config = config
        self.root   = config.DATASET_ROOT

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    @staticmethod
    def _parse_filename(filename):
        """Extract (action, subject, trial) from a*_s*_t*_<mod>.<ext>"""
        base  = os.path.basename(filename)
        parts = base.split('_')
        return int(parts[0][1:]), int(parts[1][1:]), int(parts[2][1:])

    @staticmethod
    def _build_key(action, subject, trial):
        return f"a{action}_s{subject}_t{trial}"

    def _ensure_entry(self, registry, action, subject, trial):
        key = self._build_key(action, subject, trial)
        if key not in registry:
            registry[key] = {
                'action':        action,
                'subject':       subject,
                'trial':         trial,
                'skeleton_path': None,
                'inertial_path': None,
                'depth_path':    None,
                'video_path':    None,
            }
        return key

    # ------------------------------------------------------------------
    # Stage 1 – registry builder  (no I/O on data files)
    # ------------------------------------------------------------------

    def build_sample_registry(self):
        """
        Scan modality directories and return a filename-only registry.
        No raw data is loaded.

        Returns:
            dict  { sample_key -> metadata + file paths }
        """
        registry = {}
        n_skel = n_iner = n_depth = n_video = 0

        # --- Skeleton ---
        if self.config.USE_SKELETON:
            skel_dir = os.path.join(self.root, self.config.SKELETON_DIR)
            for fpath in sorted(glob.glob(os.path.join(skel_dir, "a*_s*_t*_skeleton.mat"))):
                try:
                    a, s, t = self._parse_filename(fpath)
                    key = self._ensure_entry(registry, a, s, t)
                    registry[key]['skeleton_path'] = fpath
                    n_skel += 1
                except Exception as e:
                    logger.debug(f"Skipping skeleton file {fpath}: {e}")

        # --- Inertial ---
        if self.config.USE_INERTIAL:
            iner_dir = os.path.join(self.root, self.config.INERTIAL_DIR)
            for fpath in sorted(glob.glob(os.path.join(iner_dir, "a*_s*_t*_inertial.mat"))):
                try:
                    a, s, t = self._parse_filename(fpath)
                    key = self._ensure_entry(registry, a, s, t)
                    registry[key]['inertial_path'] = fpath
                    n_iner += 1
                except Exception as e:
                    logger.debug(f"Skipping inertial file {fpath}: {e}")

        # --- Depth ---
        if self.config.USE_DEPTH:
            depth_dir = os.path.join(self.root, self.config.DEPTH_DIR)
            for fpath in sorted(glob.glob(os.path.join(depth_dir, "a*_s*_t*_depth.mat"))):
                try:
                    a, s, t = self._parse_filename(fpath)
                    key = self._ensure_entry(registry, a, s, t)
                    registry[key]['depth_path'] = fpath
                    n_depth += 1
                except Exception as e:
                    logger.debug(f"Skipping depth file {fpath}: {e}")

        # --- Video ---
        if self.config.USE_VIDEO:
            for vdir in self.config.VIDEO_DIRS:
                video_dir = os.path.join(self.root, vdir)
                for fpath in sorted(glob.glob(os.path.join(video_dir, "a*_s*_t*_color.avi"))):
                    try:
                        a, s, t = self._parse_filename(fpath)
                        key = self._ensure_entry(registry, a, s, t)
                        registry[key]['video_path'] = fpath
                        n_video += 1
                    except Exception as e:
                        logger.debug(f"Skipping video file {fpath}: {e}")

        logger.info(
            f"Registry built – {len(registry)} unique samples | "
            f"skeleton:{n_skel}  inertial:{n_iner}  depth:{n_depth}  video:{n_video}"
        )
        return registry

    # ------------------------------------------------------------------
    # Stage 2 – on-demand per-file loaders
    # ------------------------------------------------------------------

    @staticmethod
    def load_skeleton_from_file(fpath):
        """
        Load one skeleton .mat file → (frames, 20, 3) float64, or None.
        """
        try:
            mat  = loadmat(fpath)
            skel = mat.get('d_skel', mat.get('d_skeleton', None))
            if skel is None:
                skel = next((mat[k] for k in mat if not k.startswith('__')), None)
            if skel is None:
                return None
            skel = np.array(skel, dtype=np.float64)
            if skel.ndim == 2 and skel.shape[1] == 60:
                skel = skel.reshape(-1, 20, 3)
            elif skel.ndim == 3 and skel.shape[2] == 20:
                skel = skel.transpose(2, 1, 0)   # (3, 20, F) -> (F, 20, 3)
            return skel
        except Exception as e:
            logger.debug(f"load_skeleton_from_file({fpath}): {e}")
            return None

    @staticmethod
    def load_inertial_from_file(fpath):
        """
        Load one inertial .mat file → (samples, 6) float64, or None.
        """
        try:
            mat  = loadmat(fpath)
            iner = mat.get('d_iner', mat.get('d_inertial', None))
            if iner is None:
                iner = next((mat[k] for k in mat if not k.startswith('__')), None)
            return np.array(iner, dtype=np.float64) if iner is not None else None
        except Exception as e:
            logger.debug(f"load_inertial_from_file({fpath}): {e}")
            return None

    @staticmethod
    def load_depth_from_file(fpath):
        """
        Load one depth .mat file → (H, W, frames) float64, or None.
        """
        try:
            mat   = loadmat(fpath)
            depth = mat.get('d_depth', None)
            if depth is None:
                depth = next((mat[k] for k in mat if not k.startswith('__')), None)
            return np.array(depth, dtype=np.float64) if depth is not None else None
        except Exception as e:
            logger.debug(f"load_depth_from_file({fpath}): {e}")
            return None


In [4]:
# =============================================================================
# SECTION 2b: SIMULATED RAW DATA LOSS – per-sample helpers
# =============================================================================
#
# Previously apply_raw_data_loss() received the entire samples dict and
# modified everything in-place before feature extraction, which forced all
# raw arrays to stay resident simultaneously.
#
# Now each modality has its own helper called inside the streaming loop:
#
#   apply_skeleton_loss(skel,  rate, rng)  -> ndarray (modified in-place)
#   apply_inertial_loss(iner,  rate, rng)  -> ndarray (modified in-place)
#   apply_depth_loss(depth,    rate, rng)  -> ndarray (modified in-place)
#   (RGB video loss remains inside VideoFeatureExtractor._read_video_frames)
#
# Each modality has its own RNG instance seeded from config.LOSS_SEED so
# that the corruption pattern is deterministic and reproducible, but
# independent across modalities.
# =============================================================================

import math
import gc


def _n_units_to_zero(n_units: int, rate: float) -> int:
    """ceil(n_units * rate), at least 1."""
    return max(1, math.ceil(n_units * rate))


def apply_skeleton_loss(skel: np.ndarray, rate: float,
                        rng: np.random.RandomState) -> np.ndarray:
    """
    Zero out random joints in a skeleton sequence IN-PLACE.

    Args:
        skel : (frames, n_joints, 3)
        rate : fraction of joints to zero  (e.g. 0.20)
        rng  : caller-owned RandomState (state advances each call)
    Returns:
        same array (for chaining)
    """
    if skel is None or skel.ndim != 3 or skel.shape[1] == 0:
        return skel
    n_joints = skel.shape[1]
    n_zero   = _n_units_to_zero(n_joints, rate)
    joints   = rng.choice(n_joints, size=n_zero, replace=False)
    skel[:, joints, :] = 0.0
    logger.debug(f"Skeleton loss: zeroed joints {joints.tolist()} ({n_zero}/{n_joints})")
    return skel


def apply_inertial_loss(iner: np.ndarray, rate: float,
                        rng: np.random.RandomState) -> np.ndarray:
    """
    Zero out random channels in an inertial sequence IN-PLACE.

    Args:
        iner : (time_steps, n_channels)
        rate : fraction of channels to zero
        rng  : caller-owned RandomState
    """
    if iner is None or iner.ndim != 2 or iner.shape[1] == 0:
        return iner
    n_ch   = iner.shape[1]
    n_zero = _n_units_to_zero(n_ch, rate)
    chs    = rng.choice(n_ch, size=n_zero, replace=False)
    iner[:, chs] = 0.0
    logger.debug(f"Inertial loss: zeroed channels {chs.tolist()} ({n_zero}/{n_ch})")
    return iner


def apply_depth_loss(depth: np.ndarray, rate: float,
                     rng: np.random.RandomState) -> np.ndarray:
    """
    Zero out random frames in a depth volume IN-PLACE.

    Handles both (H, W, frames) and (frames, H, W) axis orderings.

    Args:
        depth : 3-D depth array
        rate  : fraction of frames to zero
        rng   : caller-owned RandomState
    """
    if depth is None or depth.ndim != 3:
        return depth
    sh = depth.shape
    if sh[2] <= sh[0] and sh[2] <= sh[1]:     # (H, W, frames)
        n_frames = sh[2]
        n_zero   = _n_units_to_zero(n_frames, rate)
        frs      = rng.choice(n_frames, size=n_zero, replace=False)
        depth[:, :, frs] = 0.0
    else:                                       # (frames, H, W)
        n_frames = sh[0]
        n_zero   = _n_units_to_zero(n_frames, rate)
        frs      = rng.choice(n_frames, size=n_zero, replace=False)
        depth[frs, :, :] = 0.0
    logger.debug(f"Depth loss: zeroed {n_zero}/{n_frames} frames")
    return depth


In [5]:
# =============================================================================
# SECTION 3: SKELETON FEATURE EXTRACTION
# =============================================================================
#
# Two dataset-specific extractors that produce IDENTICAL feature dimensions
# and semantics.  Only the raw joint-index constants differ because UTD-MHAD
# (Kinect v1, 20 joints) and CZU-MHAD (Kinect v2, 25 joints) number their
# joints differently.
#
# Joint correspondence (the 20 shared body parts):
#
#   Body part            UTD-MHAD   CZU-MHAD
#   ─────────────────    ────────   ────────
#   head                    0          3
#   shoulder center/neck    1          2
#   spine / spine mid       2          1
#   hip center/spine base   3          0
#   left shoulder           4          4
#   left elbow              5          5
#   left wrist              6          6
#   left hand               7          7
#   right shoulder          8          8
#   right elbow             9          9
#   right wrist            10         10
#   right hand             11         11
#   left hip               12         12
#   left knee              13         13
#   left ankle             14         14
#   left foot              15         15
#   right hip              16         16
#   right knee             17         17
#   right ankle            18         18
#   right foot             19         19
#
# CZU-MHAD additionally has joints 20-24 (spine shoulder, hand tips, thumbs)
# which are NOT used in feature extraction, so both extractors output the
# exact same feature vector length.
# =============================================================================


class _SkeletonFeatureExtractorBase:
    """
    Base class containing all shared feature-extraction logic.

    Subclasses only need to set the joint-index class attributes for their
    specific dataset.  All feature computation references these semantic
    names, so the resulting feature vectors have identical meaning and
    dimensionality across datasets.

    Preprocessing pipeline:
        1. Detect missing joints (all-zero, NaN, Inf)
        2. Temporal linear interpolation to fill gaps
        3. Light Gaussian smoothing (sigma=1) to reduce sensor jitter

    Feature categories:
        1. Normalized joint position statistics  (VISTA-inspired)
        2. Pairwise joint distance statistics
        3. Joint angle statistics
        4. Bone (segment) length statistics
        5. Velocity statistics
        6. Acceleration statistics
        7. Covariance matrix features           (MSDFE-inspired)
        8. Centre-of-mass trajectory statistics
        9. Global motion energy
    """

    # --- Subclasses MUST override these ---
    HEAD = None
    SHOULDER_CENTER = None   # neck in CZU-MHAD
    SPINE = None             # spine_mid in CZU-MHAD
    HIP_CENTER = None        # spine_base in CZU-MHAD
    SHOULDER_LEFT = None
    ELBOW_LEFT = None
    WRIST_LEFT = None
    HAND_LEFT = None
    SHOULDER_RIGHT = None
    ELBOW_RIGHT = None
    WRIST_RIGHT = None
    HAND_RIGHT = None
    HIP_LEFT = None
    KNEE_LEFT = None
    ANKLE_LEFT = None
    FOOT_LEFT = None
    HIP_RIGHT = None
    KNEE_RIGHT = None
    ANKLE_RIGHT = None
    FOOT_RIGHT = None

    NUM_JOINTS = None        # 20 for UTD, 20 used (out of 25) for CZU
    RAW_JOINTS = None        # total joints in the raw data (20 or 25)
    RAW_COLS = None          # expected flat columns (60 or 75)

    def __init__(self):
        # ---- Built from semantic names (resolved at subclass import) ----
        self.JOINT_PAIRS = [
            (self.HAND_LEFT,     self.HAND_RIGHT),      # left hand – right hand
            (self.FOOT_LEFT,     self.FOOT_RIGHT),      # left foot – right foot
            (self.HAND_LEFT,     self.HIP_CENTER),      # left hand – hip
            (self.HAND_RIGHT,    self.HIP_CENTER),      # right hand – hip
            (self.HAND_LEFT,     self.HEAD),             # left hand – head
            (self.HAND_RIGHT,    self.HEAD),             # right hand – head
            (self.FOOT_LEFT,     self.HIP_CENTER),      # left foot – hip
            (self.FOOT_RIGHT,    self.HIP_CENTER),      # right foot – hip
            (self.WRIST_LEFT,    self.WRIST_RIGHT),     # left wrist – right wrist
            (self.ELBOW_LEFT,    self.ELBOW_RIGHT),     # left elbow – right elbow
            (self.SHOULDER_LEFT, self.SHOULDER_RIGHT),  # left shoulder – right shoulder
            (self.KNEE_LEFT,     self.KNEE_RIGHT),      # left knee – right knee
            (self.HIP_LEFT,      self.HIP_RIGHT),       # left hip – right hip
            (self.HAND_LEFT,     self.FOOT_LEFT),       # left hand – left foot
            (self.HAND_RIGHT,    self.FOOT_RIGHT),      # right hand – right foot
            (self.HEAD,          self.HIP_CENTER),      # head – hip (body height proxy)
        ]

        self.ANGLE_TRIPLETS = [
            (self.SHOULDER_LEFT,  self.ELBOW_LEFT,     self.WRIST_LEFT),    # L shoulder-elbow-wrist
            (self.SHOULDER_RIGHT, self.ELBOW_RIGHT,    self.WRIST_RIGHT),   # R shoulder-elbow-wrist
            (self.HIP_LEFT,       self.KNEE_LEFT,      self.ANKLE_LEFT),    # L hip-knee-ankle
            (self.HIP_RIGHT,      self.KNEE_RIGHT,     self.ANKLE_RIGHT),   # R hip-knee-ankle
            (self.SHOULDER_LEFT,  self.SHOULDER_CENTER, self.SHOULDER_RIGHT),# L shoulder – center – R shoulder
            (self.ELBOW_LEFT,     self.SHOULDER_LEFT,   self.SHOULDER_CENTER),
            (self.ELBOW_RIGHT,    self.SHOULDER_RIGHT,  self.SHOULDER_CENTER),
            (self.HIP_CENTER,     self.SPINE,           self.SHOULDER_CENTER),# hip – spine – shoulder center
            (self.HIP_LEFT,       self.HIP_CENTER,      self.HIP_RIGHT),     # L hip – center – R hip
            (self.SHOULDER_LEFT,  self.SHOULDER_CENTER,  self.HEAD),          # L shoulder – center – head
            (self.SHOULDER_RIGHT, self.SHOULDER_CENTER,  self.HEAD),          # R shoulder – center – head
        ]

        self.BONE_PAIRS = [
            (self.HIP_CENTER,     self.SPINE),
            (self.SPINE,          self.SHOULDER_CENTER),
            (self.SHOULDER_CENTER,self.HEAD),
            (self.SHOULDER_CENTER,self.SHOULDER_LEFT),
            (self.SHOULDER_LEFT,  self.ELBOW_LEFT),
            (self.ELBOW_LEFT,     self.WRIST_LEFT),
            (self.WRIST_LEFT,     self.HAND_LEFT),
            (self.SHOULDER_CENTER,self.SHOULDER_RIGHT),
            (self.SHOULDER_RIGHT, self.ELBOW_RIGHT),
            (self.ELBOW_RIGHT,    self.WRIST_RIGHT),
            (self.WRIST_RIGHT,    self.HAND_RIGHT),
            (self.HIP_CENTER,     self.HIP_LEFT),
            (self.HIP_LEFT,       self.KNEE_LEFT),
            (self.KNEE_LEFT,      self.ANKLE_LEFT),
            (self.ANKLE_LEFT,     self.FOOT_LEFT),
            (self.HIP_CENTER,     self.HIP_RIGHT),
            (self.HIP_RIGHT,      self.KNEE_RIGHT),
            (self.KNEE_RIGHT,     self.ANKLE_RIGHT),
            (self.ANKLE_RIGHT,    self.FOOT_RIGHT),
        ]

        # Key joints for position / velocity / acceleration stats
        self.KEY_JOINTS = [
            self.HIP_CENTER, self.SPINE, self.SHOULDER_CENTER, self.HEAD,
            self.SHOULDER_LEFT, self.ELBOW_LEFT, self.WRIST_LEFT, self.HAND_LEFT,
            self.SHOULDER_RIGHT, self.ELBOW_RIGHT, self.WRIST_RIGHT, self.HAND_RIGHT,
            self.FOOT_LEFT, self.FOOT_RIGHT,
        ]

        # Key joints for covariance matrix
        self.COV_JOINTS = [
            self.HIP_CENTER, self.SPINE, self.SHOULDER_CENTER, self.HEAD,
            self.HAND_LEFT, self.HAND_RIGHT, self.FOOT_LEFT, self.FOOT_RIGHT,
        ]

    # =====================================================================
    # Preprocessing
    # =====================================================================

    def _detect_missing(self, skel):
        """
        Detect missing / invalid joint readings.
        Missing joints are encoded as all-zero coordinates, NaN, or Inf.

        Args:
            skel: (frames, J, 3)
        Returns:
            boolean mask (frames, J), True = missing
        """
        is_nan = np.any(np.isnan(skel), axis=2)
        is_inf = np.any(np.isinf(skel), axis=2)
        is_zero = np.all(np.abs(skel) < 1e-10, axis=2)
        return is_nan | is_inf | is_zero

    def _interpolate_missing(self, skel, missing_mask):
        """
        Fill missing joints via linear temporal interpolation.

        - Interior gaps: linear interp between nearest valid frames
        - Leading / trailing gaps: forward / backward fill (nearest valid)
        - Joint missing in ALL frames: copy from kinematic parent, else zero

        Args:
            skel: (frames, J, 3)
            missing_mask: (frames, J) boolean
        Returns:
            skel with gaps filled
        """
        skel = skel.copy()
        n_frames, n_joints, _ = skel.shape

        for j in range(n_joints):
            if not np.any(missing_mask[:, j]):
                continue

            valid_idx = np.where(~missing_mask[:, j])[0]
            if len(valid_idx) == 0:
                continue  # handled below

            for axis in range(3):
                skel[:, j, axis] = np.interp(
                    np.arange(n_frames), valid_idx, skel[valid_idx, j, axis]
                )

        # Joints missing in ALL frames: copy from kinematic parent
        all_missing = np.where(np.all(missing_mask, axis=0))[0]
        if len(all_missing) > 0:
            parent_map = {}
            for p, c in self.BONE_PAIRS:
                parent_map[c] = p
            for j in all_missing:
                parent = parent_map.get(j, None)
                if parent is not None and not np.all(missing_mask[:, parent]):
                    skel[:, j, :] = skel[:, parent, :]
                else:
                    skel[:, j, :] = 0.0

        return skel

    def _preprocess(self, skel):
        """
        Full preprocessing: detect missing → interpolate → smooth.

        Args:
            skel: (frames, J, 3)
        Returns:
            cleaned skeleton (frames, J, 3)
        """
        missing_mask = self._detect_missing(skel)
        skel = np.nan_to_num(skel, nan=0.0, posinf=0.0, neginf=0.0)

        n_missing = np.sum(missing_mask)
        if n_missing > 0:
            logger.debug(
                f"Skeleton preprocessing: {n_missing} missing joint-frames "
                f"({100 * n_missing / missing_mask.size:.1f}%) — interpolating"
            )
            skel = self._interpolate_missing(skel, missing_mask)

        # Light temporal Gaussian smoothing to reduce sensor jitter
        if skel.shape[0] >= 5:
            from scipy.ndimage import gaussian_filter1d
            skel = gaussian_filter1d(skel, sigma=1.0, axis=0)

        return skel

    # =====================================================================
    # Normalization
    # =====================================================================

    def _normalize_skeleton(self, skel):
        """
        Translate to hip center, scale by torso length (hip → shoulder center).

        Args:
            skel: (frames, J, 3)
        Returns:
            normalized skeleton (frames, J, 3)
        """
        hip = skel[:, self.HIP_CENTER:self.HIP_CENTER + 1, :]
        skel_norm = skel - hip

        torso_vec = (skel_norm[:, self.SHOULDER_CENTER, :]
                     - skel_norm[:, self.HIP_CENTER, :])
        torso_len = np.linalg.norm(torso_vec, axis=1, keepdims=True)
        torso_len = np.clip(torso_len, 1e-6, None)
        skel_norm = skel_norm / torso_len[:, np.newaxis, :]

        return skel_norm

    # =====================================================================
    # Feature helpers
    # =====================================================================

    def _compute_joint_distances(self, skel):
        """Pairwise joint distances over time. → (frames, 16)"""
        distances = []
        for j1, j2 in self.JOINT_PAIRS:
            distances.append(np.linalg.norm(skel[:, j1, :] - skel[:, j2, :], axis=1))
        return np.array(distances).T

    def _compute_joint_angles(self, skel):
        """Joint angles at triplet vertices. → (frames, 11)"""
        angles = []
        for j1, j2, j3 in self.ANGLE_TRIPLETS:
            v1 = skel[:, j1, :] - skel[:, j2, :]
            v2 = skel[:, j3, :] - skel[:, j2, :]
            cos_a = np.sum(v1 * v2, axis=1) / (
                np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1) + 1e-8
            )
            angles.append(np.arccos(np.clip(cos_a, -1.0, 1.0)))
        return np.array(angles).T

    def _compute_bone_lengths(self, skel):
        """Bone segment lengths over time. → (frames, 19)"""
        lengths = []
        for j1, j2 in self.BONE_PAIRS:
            lengths.append(np.linalg.norm(skel[:, j1, :] - skel[:, j2, :], axis=1))
        return np.array(lengths).T

    def _compute_velocity(self, skel):
        if skel.shape[0] < 2:
            return np.zeros_like(skel)
        vel = np.diff(skel, axis=0)
        return np.vstack([vel, vel[-1:]])

    def _compute_acceleration(self, skel):
        vel = self._compute_velocity(skel)
        if vel.shape[0] < 2:
            return np.zeros_like(vel)
        acc = np.diff(vel, axis=0)
        return np.vstack([acc, acc[-1:]])

    def _temporal_statistics(self, signal_2d):
        """
        9 statistics per column: mean, std, RMS, skew, kurtosis,
        range, median, Q1, Q3.
        """
        features = []
        for col in range(signal_2d.shape[1]):
            s = signal_2d[:, col]
            features.extend([
                np.mean(s),
                np.std(s),
                np.sqrt(np.mean(s ** 2)),
                stats.skew(s) if len(s) > 2 else 0.0,
                stats.kurtosis(s) if len(s) > 3 else 0.0,
                np.max(s) - np.min(s),
                np.median(s),
                np.percentile(s, 25),
                np.percentile(s, 75),
            ])
        return np.array(features)

    def _covariance_features(self, skel):
        """Upper-triangle of covariance matrix over 8 key joints (24 coords)."""
        flat = skel.reshape(skel.shape[0], -1)
        idx = []
        for j in self.COV_JOINTS:
            idx.extend([j * 3, j * 3 + 1, j * 3 + 2])
        flat_key = flat[:, idx]

        if flat_key.shape[0] < 2:
            cov = np.zeros((flat_key.shape[1], flat_key.shape[1]))
        else:
            cov = np.cov(flat_key.T)

        return cov[np.triu_indices(cov.shape[0])]

    # =====================================================================
    # Reshape helpers (dataset-specific override if needed)
    # =====================================================================

    def _to_frames_joints_3(self, skel):
        """
        Convert any common skeleton layout to (frames, J, 3).
        Handles (frames, J*3), (J, 3, frames), etc.

        Returns (frames, J, 3) array or None if shape is unrecognisable.
        """
        if skel.ndim == 2:
            n_frames, n_cols = skel.shape
            if n_cols >= self.RAW_COLS:
                return skel[:, :self.RAW_COLS].reshape(n_frames, self.RAW_JOINTS, 3)
            return None

        if skel.ndim != 3:
            return None

        s = skel.shape
        J = self.RAW_JOINTS

        if s[1] == J and s[2] == 3:
            return skel
        if s[0] == J and s[2] == 3:
            return np.transpose(skel, (1, 0, 2))
        if s[2] == J and s[1] == 3:
            return np.transpose(skel, (0, 2, 1))
        if s[0] == J and s[1] == 3:
            return np.transpose(skel, (2, 0, 1))
        if s[0] == 3 and s[1] == J:
            return np.transpose(skel, (2, 1, 0))
        if s[2] == J:
            return np.transpose(skel, (2, 0, 1))

        # Last resort
        return np.transpose(skel, (2, 0, 1))

    # =====================================================================
    # Main extraction
    # =====================================================================

    def extract(self, skel):
        """
        Extract skeleton feature vector from a single sequence.

        Accepts raw data in various shapes; reshapes, preprocesses,
        normalises, then extracts all feature categories.

        Feature dimension breakdown (both datasets identical):
            pos_feats:     14 key joints × 3 coords × 9 stats = 378
            dist_feats:    16 pairs × 9 stats                 = 144
            angle_feats:   11 triplets × 9 stats               =  99
            bone_feats:    19 bones × 9 stats                  = 171
            vel_feats:     14 × 3 × 9                          = 378
            acc_feats:     14 × 3 × 9                          = 378
            cov_feats:     upper tri of 24×24                  = 300
            com_feats:     3 × 9                               =  27
            motion_energy: 4                                   =   4
            ─────────────────────────────────────────────────────────
            TOTAL                                              = 1879

        Args:
            skel: skeleton data in any common layout
        Returns:
            1D numpy array of length 1879, or None
        """
        if skel is None or skel.size == 0:
            return None

        skel = self._to_frames_joints_3(skel)
        if skel is None or skel.shape[0] == 0:
            return None

        # If CZU-MHAD 25 joints, keep only the 20 shared joints
        if skel.shape[1] > 20:
            skel = skel[:, :20, :]

        # --- Preprocessing ---
        skel = self._preprocess(skel)

        # --- Normalization ---
        skel_norm = self._normalize_skeleton(skel)

        # --- Features ---
        # 1. Position statistics on key joints
        flat_pos = skel_norm.reshape(skel_norm.shape[0], -1)
        key_idx = []
        for j in self.KEY_JOINTS:
            key_idx.extend([j * 3, j * 3 + 1, j * 3 + 2])
        pos_feats = self._temporal_statistics(flat_pos[:, key_idx])

        # 2. Joint distances
        dist_feats = self._temporal_statistics(self._compute_joint_distances(skel_norm))

        # 3. Joint angles
        angle_feats = self._temporal_statistics(self._compute_joint_angles(skel_norm))

        # 4. Bone lengths
        bone_feats = self._temporal_statistics(self._compute_bone_lengths(skel_norm))

        # 5. Velocity
        vel = self._compute_velocity(skel_norm).reshape(skel_norm.shape[0], -1)
        vel_feats = self._temporal_statistics(vel[:, key_idx])

        # 6. Acceleration
        acc = self._compute_acceleration(skel_norm).reshape(skel_norm.shape[0], -1)
        acc_feats = self._temporal_statistics(acc[:, key_idx])

        # 7. Covariance
        cov_feats = self._covariance_features(skel_norm)

        # 8. Centre of mass
        com = np.mean(skel_norm, axis=1)
        com_feats = self._temporal_statistics(com)

        # 9. Motion energy
        if skel_norm.shape[0] > 1:
            total_disp = np.sum(
                np.linalg.norm(np.diff(skel_norm, axis=0), axis=2), axis=1
            )
        else:
            total_disp = np.array([0.0])
        motion_energy = np.array([
            np.mean(total_disp), np.std(total_disp),
            np.max(total_disp), np.sum(total_disp),
        ])

        all_feats = np.concatenate([
            pos_feats, dist_feats, angle_feats, bone_feats,
            vel_feats, acc_feats, cov_feats, com_feats, motion_energy,
        ])

        return np.nan_to_num(all_feats, nan=0.0, posinf=0.0, neginf=0.0)


# =========================================================================
# UTD-MHAD  (Kinect v1, 20 joints)
# =========================================================================

class SkeletonFeatureExtractor(_SkeletonFeatureExtractorBase):
    """
    Skeleton feature extractor for the UTD-MHAD dataset.

    UTD-MHAD joint indices (Kinect v1, 20 joints):
        0: head, 1: shoulder center, 2: spine, 3: hip center,
        4: left shoulder, 5: left elbow, 6: left wrist, 7: left hand,
        8: right shoulder, 9: right elbow, 10: right wrist, 11: right hand,
        12: left hip, 13: left knee, 14: left ankle, 15: left foot,
        16: right hip, 17: right knee, 18: right ankle, 19: right foot
    """
    HEAD             = 0
    SHOULDER_CENTER  = 1
    SPINE            = 2
    HIP_CENTER       = 3
    SHOULDER_LEFT    = 4
    ELBOW_LEFT       = 5
    WRIST_LEFT       = 6
    HAND_LEFT        = 7
    SHOULDER_RIGHT   = 8
    ELBOW_RIGHT      = 9
    WRIST_RIGHT      = 10
    HAND_RIGHT       = 11
    HIP_LEFT         = 12
    KNEE_LEFT        = 13
    ANKLE_LEFT       = 14
    FOOT_LEFT        = 15
    HIP_RIGHT        = 16
    KNEE_RIGHT       = 17
    ANKLE_RIGHT      = 18
    FOOT_RIGHT       = 19

    NUM_JOINTS = 20
    RAW_JOINTS = 20
    RAW_COLS   = 60   # 20 × 3

In [6]:
# =============================================================================
# SECTION 4: INERTIAL FEATURE EXTRACTION
# =============================================================================

class InertialFeatureExtractor:
    """
    Extracts features from inertial sensor data (accelerometer + gyroscope).

    Directly inspired by the VISTA paper (Fiorini et al., 2022):
    - Time-domain features: mean, std, RMS, skewness, kurtosis, SMA, power
    - Additional: zero-crossing rate, peak count, signal energy
    - Frequency-domain: dominant frequency, spectral entropy, band energy

    The VISTA paper demonstrated these features on wrist + finger IMUs
    and showed 73-81% accuracy with individual sensors and higher with fusion.
    """

    def __init__(self, config):
        self.window_size = config.INERTIAL_WINDOW_SIZE
        self.overlap = config.INERTIAL_WINDOW_OVERLAP

    def _signal_magnitude_area(self, data):
        """
        Signal Magnitude Area (SMA) - used in VISTA paper.
        SMA = (1/N) * sum(|ax| + |ay| + |az|)
        """
        return np.mean(np.sum(np.abs(data), axis=1))

    def _signal_power(self, s):
        """Signal power = mean of squared values."""
        return np.mean(s**2)

    def _zero_crossing_rate(self, s):
        """Count zero crossings normalized by length."""
        s_centered = s - np.mean(s)
        return np.sum(np.abs(np.diff(np.sign(s_centered)))) / (2.0 * len(s))

    def _peak_count(self, s):
        """Count number of peaks in signal."""
        peaks, _ = signal.find_peaks(s)
        return len(peaks) / len(s)

    def _spectral_entropy(self, s, fs=50):
        """Compute spectral entropy of the signal."""
        freqs, psd = signal.welch(s, fs=fs, nperseg=min(len(s), 256))
        psd_norm = psd / (np.sum(psd) + 1e-12)
        psd_norm = psd_norm[psd_norm > 0]
        return -np.sum(psd_norm * np.log2(psd_norm + 1e-12))

    def _dominant_frequency(self, s, fs=50):
        """Find the dominant frequency component."""
        if len(s) < 4:
            return 0.0
        freqs, psd = signal.welch(s, fs=fs, nperseg=min(len(s), 256))
        return freqs[np.argmax(psd)]

    def _frequency_band_energy(self, s, fs=50, bands=[(0, 5), (5, 15), (15, 25)]):
        """Energy in different frequency bands."""
        if len(s) < 4:
            return [0.0] * len(bands)
        freqs, psd = signal.welch(s, fs=fs, nperseg=min(len(s), 256))
        energies = []
        for low, high in bands:
            mask = (freqs >= low) & (freqs < high)
            energies.append(np.sum(psd[mask]))
        return energies

    def _extract_channel_features(self, s):
        """
        Extract comprehensive features from a single channel.
        Based on VISTA paper's feature set + extensions.
        """
        features = []

        # Time-domain (VISTA paper features)
        features.append(np.mean(s))                          # Mean
        features.append(np.std(s))                           # Standard deviation
        features.append(np.sqrt(np.mean(s**2)))              # RMS
        features.append(stats.skew(s) if len(s) > 2 else 0) # Skewness
        features.append(stats.kurtosis(s) if len(s) > 3 else 0)  # Kurtosis
        features.append(self._signal_power(s))               # Power
        features.append(np.max(s) - np.min(s))               # Range
        features.append(np.median(s))                        # Median
        features.append(np.mean(np.abs(s)))                  # Mean absolute value
        features.append(np.percentile(s, 25))                # Q1
        features.append(np.percentile(s, 75))                # Q3
        features.append(np.percentile(s, 75) - np.percentile(s, 25))  # IQR
        features.append(self._zero_crossing_rate(s))         # Zero-crossing rate
        features.append(self._peak_count(s))                 # Peak count

        # Frequency-domain
        features.append(self._dominant_frequency(s))         # Dominant freq
        features.append(self._spectral_entropy(s))           # Spectral entropy
        features.extend(self._frequency_band_energy(s))      # Band energies

        return features

    def _extract_window_features(self, window):
        """
        Extract features from a single window of inertial data.

        Args:
            window: (window_size, 6) array [acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z]
        Returns:
            1D feature vector
        """
        features = []

        acc = window[:, :3]   # Accelerometer
        gyro = window[:, 3:]  # Gyroscope

        # Per-channel features for all 6 channels
        for ch in range(6):
            features.extend(self._extract_channel_features(window[:, ch]))

        # Accelerometer magnitude
        acc_mag = np.linalg.norm(acc, axis=1)
        features.extend(self._extract_channel_features(acc_mag))

        # Gyroscope magnitude
        gyro_mag = np.linalg.norm(gyro, axis=1)
        features.extend(self._extract_channel_features(gyro_mag))

        # Signal Magnitude Area (SMA) - key VISTA feature
        features.append(self._signal_magnitude_area(acc))
        features.append(self._signal_magnitude_area(gyro))

        # Cross-axis correlations (inspired by MSDFE covariance idea)
        for i in range(3):
            for j in range(i+1, 3):
                if len(acc[:, i]) > 1:
                    corr = np.corrcoef(acc[:, i], acc[:, j])[0, 1]
                    features.append(corr if not np.isnan(corr) else 0.0)
                else:
                    features.append(0.0)

        for i in range(3):
            for j in range(i+1, 3):
                if len(gyro[:, i]) > 1:
                    corr = np.corrcoef(gyro[:, i], gyro[:, j])[0, 1]
                    features.append(corr if not np.isnan(corr) else 0.0)
                else:
                    features.append(0.0)

        # Acc-Gyro cross-correlations
        for i in range(3):
            if len(acc[:, i]) > 1:
                corr = np.corrcoef(acc[:, i], gyro[:, i])[0, 1]
                features.append(corr if not np.isnan(corr) else 0.0)
            else:
                features.append(0.0)

        return features

    def extract(self, inertial_data):
        """
        Extract feature vector from a full inertial sequence.

        Strategy: segment into overlapping windows, extract features per window,
        then aggregate window features with statistics.

        Args:
            inertial_data: (samples, 6) array
        Returns:
            1D feature vector
        """
        if inertial_data is None or inertial_data.shape[0] == 0:
            return None

        inertial_data = np.nan_to_num(inertial_data, nan=0.0, posinf=0.0, neginf=0.0)

        # Apply low-pass Butterworth filter (VISTA paper uses 5Hz cutoff)
        try:
            b, a = signal.butter(4, 0.2, btype='low')  # Normalized frequency
            for ch in range(inertial_data.shape[1]):
                if len(inertial_data[:, ch]) > 12:
                    inertial_data[:, ch] = signal.filtfilt(b, a, inertial_data[:, ch])
        except Exception:
            pass  # Skip filtering if it fails

        # Segment into windows
        n_samples = inertial_data.shape[0]
        step = int(self.window_size * (1 - self.overlap))
        step = max(step, 1)

        windows = []
        for start in range(0, n_samples - self.window_size + 1, step):
            windows.append(inertial_data[start:start + self.window_size])

        if not windows:
            # If sequence is shorter than window, use entire sequence
            windows = [inertial_data]

        # Extract features from each window
        window_features = []
        for w in windows:
            wf = self._extract_window_features(w)
            window_features.append(wf)

        window_features = np.array(window_features)

        # Aggregate: compute statistics across windows
        if window_features.shape[0] == 1:
            return np.nan_to_num(window_features[0], nan=0.0, posinf=0.0, neginf=0.0)

        aggregated = []
        for col in range(window_features.shape[1]):
            col_data = window_features[:, col]
            aggregated.extend([
                np.mean(col_data),
                np.std(col_data),
                np.min(col_data),
                np.max(col_data),
            ])

        return np.nan_to_num(np.array(aggregated), nan=0.0, posinf=0.0, neginf=0.0)


In [7]:
# =============================================================================
# SECTION 5: DEPTH FEATURE EXTRACTION
# =============================================================================

class DepthFeatureExtractor:
    """
    Extracts features from depth map sequences.

    Implements DMM-HOG from CZU-MHAD paper (Chao et al., 2022):
    - Depth Motion Maps (DMM): Project depth frame differences onto
      three orthogonal Cartesian planes (front, side, top)
    - HOG features extracted from each DMM projection
    
    Also retains original features (silhouette, edge, depth stats, temporal)
    for a combined representation.

    References:
    - Chao et al. (2022): CZU-MHAD dataset paper, Section IV-C
    - Chen et al. (2016): DMM method [42] in the paper
    - Yang et al. (2012): DMM-HOG [43] in the paper
    - Siddiqi & Alrashdi (2022): Edge detection features from depth maps
    """

    def __init__(self, config):
        self.n_sample_frames = config.DEPTH_SAMPLE_FRAMES
        # DMM-HOG parameters
        self.dmm_resize = (64, 64)  # Resize DMM maps for consistent HOG output
        self.hog_orientations = 9
        self.hog_pixels_per_cell = (8, 8)
        self.hog_cells_per_block = (2, 2)

    # -------------------------------------------------------------------------
    # DMM-HOG methods (from CZU-MHAD paper)
    # -------------------------------------------------------------------------

    def _compute_dmm(self, depth_data):
        """
        Compute Depth Motion Maps by projecting frame differences onto
        three orthogonal Cartesian planes (front, side, top).

        Given depth volume of shape (frames, H, W):
        - Front projection (xy): collapse along depth axis
        - Side projection (yz): collapse along x axis
        - Top projection (xz): collapse along y axis

        Args:
            depth_data: numpy array of shape (frames, H, W)
        Returns:
            dmm_front, dmm_side, dmm_top: three 2D motion maps
        """
        n_frames, H, W = depth_data.shape

        dmm_front = np.zeros((H, W), dtype=np.float64)
        dmm_side = np.zeros((H, 256), dtype=np.float64)  # H x depth_bins
        dmm_top = np.zeros((256, W), dtype=np.float64)    # depth_bins x W

        depth_bins = 256

        for i in range(1, n_frames):
            prev_frame = depth_data[i - 1].astype(np.float64)
            curr_frame = depth_data[i].astype(np.float64)
            diff = np.abs(curr_frame - prev_frame)

            # Front projection (x-y plane): accumulate absolute differences
            dmm_front += diff

            # For side and top projections, we need to bin depth values
            # Side projection (y-z plane): for each row y, accumulate motion
            # across depth bins
            for y in range(H):
                for x in range(W):
                    if diff[y, x] > 0:
                        d_curr = int(curr_frame[y, x])
                        if 0 < d_curr < depth_bins:
                            dmm_side[y, d_curr] += diff[y, x]

            # Top projection (x-z plane): for each col x, accumulate motion
            # across depth bins
            for y in range(H):
                for x in range(W):
                    if diff[y, x] > 0:
                        d_curr = int(curr_frame[y, x])
                        if 0 < d_curr < depth_bins:
                            dmm_top[d_curr, x] += diff[y, x]

        return dmm_front, dmm_side, dmm_top

    def _compute_dmm_fast(self, depth_data):
        """
        Fast vectorized computation of Depth Motion Maps.
        Projects frame differences onto three orthogonal planes.

        Args:
            depth_data: numpy array of shape (frames, H, W)
        Returns:
            dmm_front, dmm_side, dmm_top: three 2D motion maps
        """
        n_frames, H, W = depth_data.shape
        depth_bins = 256

        # Compute all frame differences at once
        diffs = np.abs(
            depth_data[1:].astype(np.float64) - depth_data[:-1].astype(np.float64)
        )

        # Front projection (x-y plane): sum of absolute differences per pixel
        dmm_front = np.sum(diffs, axis=0)

        # For side and top projections, use the current frame depth as bin index
        curr_frames = depth_data[1:].astype(np.int32)

        dmm_side = np.zeros((H, depth_bins), dtype=np.float64)
        dmm_top = np.zeros((depth_bins, W), dtype=np.float64)

        # Vectorized side projection: for each (frame, y, x) accumulate
        # diff[frame, y, x] into dmm_side[y, depth_bin]
        for f_idx in range(n_frames - 1):
            diff_frame = diffs[f_idx]
            depth_frame = curr_frames[f_idx]
            mask = (diff_frame > 0) & (depth_frame > 0) & (depth_frame < depth_bins)

            if not np.any(mask):
                continue

            ys, xs = np.where(mask)
            d_vals = depth_frame[ys, xs]
            diff_vals = diff_frame[ys, xs]

            # Side projection: accumulate into (y, depth)
            np.add.at(dmm_side, (ys, d_vals), diff_vals)
            # Top projection: accumulate into (depth, x)
            np.add.at(dmm_top, (d_vals, xs), diff_vals)

        return dmm_front, dmm_side, dmm_top

    def _resize_map(self, img, target_size):
        """
        Resize a 2D map to target_size using simple block averaging/interpolation.
        Uses numpy-only approach to avoid cv2 dependency.
        """
        h, w = img.shape
        th, tw = target_size
        # Simple nearest-neighbor resize
        row_indices = (np.arange(th) * h / th).astype(int)
        col_indices = (np.arange(tw) * w / tw).astype(int)
        row_indices = np.clip(row_indices, 0, h - 1)
        col_indices = np.clip(col_indices, 0, w - 1)
        return img[np.ix_(row_indices, col_indices)]

    def _normalize_map(self, img):
        """Normalize map to [0, 1] range."""
        min_val = np.min(img)
        max_val = np.max(img)
        if max_val - min_val > 1e-12:
            return (img - min_val) / (max_val - min_val)
        return np.zeros_like(img)

    def _compute_hog(self, img):
        """
        Compute HOG (Histogram of Oriented Gradients) features from a 2D image.
        
        Pure numpy implementation following Dalal & Triggs (2005):
        - Compute gradients using [-1, 0, 1] filters
        - Bin gradient orientations into histogram cells
        - Normalize over overlapping blocks
        
        Args:
            img: 2D numpy array (should be resized to self.dmm_resize)
        Returns:
            1D feature vector of HOG descriptors
        """
        img = self._resize_map(img, self.dmm_resize)
        img = self._normalize_map(img)

        H, W = img.shape
        n_orientations = self.hog_orientations
        pix_per_cell_y, pix_per_cell_x = self.hog_pixels_per_cell
        cells_per_block_y, cells_per_block_x = self.hog_cells_per_block

        # Compute gradients
        gx = np.zeros_like(img)
        gy = np.zeros_like(img)
        gx[:, 1:-1] = img[:, 2:] - img[:, :-2]
        gy[1:-1, :] = img[2:, :] - img[:-2, :]

        magnitude = np.sqrt(gx ** 2 + gy ** 2)
        orientation = np.arctan2(gy, gx + 1e-12)
        # Map orientation from [-pi, pi] to [0, pi] (unsigned gradients)
        orientation = orientation % np.pi

        # Compute cell histograms
        n_cells_y = H // pix_per_cell_y
        n_cells_x = W // pix_per_cell_x

        cell_hists = np.zeros((n_cells_y, n_cells_x, n_orientations))

        bin_width = np.pi / n_orientations

        for cy in range(n_cells_y):
            for cx in range(n_cells_x):
                y_start = cy * pix_per_cell_y
                y_end = y_start + pix_per_cell_y
                x_start = cx * pix_per_cell_x
                x_end = x_start + pix_per_cell_x

                cell_mag = magnitude[y_start:y_end, x_start:x_end].ravel()
                cell_ori = orientation[y_start:y_end, x_start:x_end].ravel()

                # Bin orientations with magnitude weighting
                bin_indices = (cell_ori / bin_width).astype(int)
                bin_indices = np.clip(bin_indices, 0, n_orientations - 1)

                for b in range(n_orientations):
                    cell_hists[cy, cx, b] = np.sum(cell_mag[bin_indices == b])

        # Block normalization (L2-norm)
        n_blocks_y = n_cells_y - cells_per_block_y + 1
        n_blocks_x = n_cells_x - cells_per_block_x + 1

        if n_blocks_y <= 0 or n_blocks_x <= 0:
            # Image too small for block normalization, return flattened cell hists
            return cell_hists.ravel()

        hog_features = []
        for by in range(n_blocks_y):
            for bx in range(n_blocks_x):
                block = cell_hists[
                    by : by + cells_per_block_y,
                    bx : bx + cells_per_block_x,
                    :
                ].ravel()
                norm = np.sqrt(np.sum(block ** 2) + 1e-12)
                block = block / norm
                hog_features.extend(block.tolist())

        return np.array(hog_features, dtype=np.float64)

    def _extract_dmm_hog_features(self, depth_data):
        """
        Extract DMM-HOG features from a depth sequence.
        
        Computes Depth Motion Maps on three orthogonal planes,
        then extracts HOG features from each, and concatenates.
        
        Args:
            depth_data: numpy array of shape (frames, H, W)
        Returns:
            1D feature vector (concatenated HOG from 3 DMM projections)
        """
        dmm_front, dmm_side, dmm_top = self._compute_dmm_fast(depth_data)

        hog_front = self._compute_hog(dmm_front)
        hog_side = self._compute_hog(dmm_side)
        hog_top = self._compute_hog(dmm_top)

        return np.concatenate([hog_front, hog_side, hog_top])

    # -------------------------------------------------------------------------
    # Original feature methods (retained for combined representation)
    # -------------------------------------------------------------------------

    def _get_silhouette(self, depth_frame):
        """Extract binary silhouette from depth frame."""
        valid = depth_frame[depth_frame > 0]
        if len(valid) == 0:
            return np.zeros_like(depth_frame, dtype=bool)
        threshold = np.percentile(valid, 50)
        return (depth_frame > 0) & (depth_frame < threshold)

    def _sobel_features(self, frame):
        """
        Compute Sobel edge features (inspired by Siddiqi paper).
        Edge magnitude and direction histograms.
        """
        sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float64)
        sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float64)

        from scipy.ndimage import convolve
        gx = convolve(frame.astype(np.float64), sobel_x)
        gy = convolve(frame.astype(np.float64), sobel_y)

        magnitude = np.sqrt(gx**2 + gy**2)
        direction = np.arctan2(gy, gx + 1e-12)

        features = []

        features.extend([
            np.mean(magnitude),
            np.std(magnitude),
            np.max(magnitude),
            np.sum(magnitude > np.mean(magnitude)),
        ])

        dir_hist, _ = np.histogram(direction[magnitude > np.mean(magnitude)],
                                    bins=8, range=(-np.pi, np.pi))
        if np.sum(dir_hist) > 0:
            dir_hist = dir_hist / (np.sum(dir_hist) + 1e-12)
        features.extend(dir_hist.tolist())

        mag_hist, _ = np.histogram(magnitude.ravel(), bins=8)
        if np.sum(mag_hist) > 0:
            mag_hist = mag_hist / (np.sum(mag_hist) + 1e-12)
        features.extend(mag_hist.tolist())

        return features

    def _shape_features(self, silhouette):
        """Compute shape descriptors from binary silhouette."""
        features = []

        area = np.sum(silhouette)
        features.append(area)

        if area == 0:
            return features + [0] * 7

        rows = np.any(silhouette, axis=1)
        cols = np.any(silhouette, axis=0)
        if np.any(rows) and np.any(cols):
            rmin, rmax = np.where(rows)[0][[0, -1]]
            cmin, cmax = np.where(cols)[0][[0, -1]]
            height = rmax - rmin + 1
            width = cmax - cmin + 1
            features.append(height)
            features.append(width)
            features.append(height / (width + 1e-6))
            features.append(area / (height * width + 1e-6))
        else:
            features.extend([0, 0, 0, 0])

        y_coords, x_coords = np.where(silhouette)
        if len(y_coords) > 0:
            features.append(np.mean(y_coords) / silhouette.shape[0])
            features.append(np.mean(x_coords) / silhouette.shape[1])
            features.append(np.std(y_coords) / (silhouette.shape[0] + 1e-6))
        else:
            features.extend([0, 0, 0])

        return features

    def _depth_distribution_features(self, depth_frame):
        """Statistical features from depth value distribution."""
        valid = depth_frame[depth_frame > 0].ravel()
        if len(valid) == 0:
            return [0] * 8

        features = [
            np.mean(valid),
            np.std(valid),
            np.median(valid),
            np.min(valid),
            np.max(valid),
            stats.skew(valid) if len(valid) > 2 else 0,
            stats.kurtosis(valid) if len(valid) > 3 else 0,
            np.percentile(valid, 75) - np.percentile(valid, 25),
        ]
        return features

    def _extract_frame_features(self, frame):
        """Extract features from a single depth frame."""
        features = []

        h, w = frame.shape
        scale = 4
        small = frame[::scale, ::scale]

        sil = self._get_silhouette(small)
        features.extend(self._shape_features(sil))

        features.extend(self._sobel_features(small))

        features.extend(self._depth_distribution_features(small))

        return features

    def _load_depth_from_path(self, filepath):
        """Load depth data from .mat file path (lazy loading to save memory)."""
        try:
            mat = loadmat(filepath)
            key = [k for k in mat.keys() if not k.startswith("__")][0]
            return extract_numeric_array(mat[key])
        except Exception as e:
            logger.debug(f"Error loading depth {filepath}: {e}")
            return None

    def extract(self, depth_data):
        """
        Extract feature vector from a depth sequence.

        Combines:
        1. DMM-HOG features (from CZU-MHAD paper) - captures global
           spatio-temporal motion patterns via depth motion maps projected
           onto three orthogonal planes with HOG descriptors
        2. Original per-frame features (silhouette, edge, depth stats,
           temporal) - captures frame-level shape and appearance cues

        Args:
            depth_data: filepath string (lazy load) OR (frames, H, W) array
        Returns:
            1D feature vector: [dmm_hog_features | original_aggregated_features]
            Always same length for a given config.
        """
        # Support lazy loading from file path
        if isinstance(depth_data, str):
            depth_data = self._load_depth_from_path(depth_data)

        if depth_data is None or depth_data.size == 0:
            return None

        depth_data = np.nan_to_num(depth_data, nan=0.0, posinf=0.0, neginf=0.0)

        # Ensure shape is (frames, height, width)
        if depth_data.ndim == 3:
            if depth_data.shape[0] > depth_data.shape[2]:
                depth_data = np.transpose(depth_data, (2, 0, 1))
        else:
            return None

        n_frames = depth_data.shape[0]

        if n_frames == 0:
            return None

        # -----------------------------------------------------------------
        # Part 1: DMM-HOG features (paper method)
        # Uses ALL frames for DMM computation (temporal accumulation)
        # -----------------------------------------------------------------
        dmm_hog_features = self._extract_dmm_hog_features(depth_data)

        # -----------------------------------------------------------------
        # Part 2: Original per-frame features (sampled frames)
        # -----------------------------------------------------------------
        if n_frames >= self.n_sample_frames:
            frame_indices = np.linspace(0, n_frames - 1, self.n_sample_frames, dtype=int)
        else:
            frame_indices = np.array([i % n_frames for i in range(self.n_sample_frames)])

        frame_features = []
        for idx in frame_indices:
            ff = self._extract_frame_features(depth_data[idx])
            frame_features.append(ff)

        frame_features = np.array(frame_features)

        # Temporal features from frame differences
        diffs = np.diff(frame_features, axis=0)
        temporal_feats = []
        for col in range(diffs.shape[1]):
            temporal_feats.extend([
                np.mean(diffs[:, col]),
                np.std(diffs[:, col]),
            ])

        # Aggregate frame features
        aggregated = []
        for col in range(frame_features.shape[1]):
            col_data = frame_features[:, col]
            aggregated.extend([
                np.mean(col_data),
                np.std(col_data),
                np.min(col_data),
                np.max(col_data),
            ])

        original_feats = np.concatenate([aggregated, temporal_feats])

        # -----------------------------------------------------------------
        # Combine DMM-HOG + original features
        # -----------------------------------------------------------------
        all_feats = np.concatenate([dmm_hog_features, original_feats])
        del depth_data
        return np.nan_to_num(all_feats, nan=0.0, posinf=0.0, neginf=0.0)

In [8]:
# =============================================================================
# SECTION 5b: VIDEO (RGB) FEATURE EXTRACTION
# =============================================================================

import cv2
import numpy as np
from scipy import stats
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import logging

logger = logging.getLogger(__name__)


class VideoFeatureExtractor:
    """
    Extracts features from RGB video sequences.

    Directly inspired by the VISTA paper (Fiorini et al., 2022):
    - Extracts 2D skeleton keypoints from video frames using MediaPipe Pose
    - Normalizes 2D joint coordinates relative to the torso (hip/shoulder
      midpoint), following VISTA's torso-based normalization
    - Selects a discriminative subset of body keypoints (head, neck/torso,
      hands/wrists, feet/ankles)
    - Segments into overlapping temporal windows and computes mean joint
      positions per window (VISTA used 3-second windows with 50% overlap)
    - Extracts temporal statistics: mean, std, RMS, skewness, kurtosis,
      range, median, Q1, Q3 over the full joint trajectory sequence

    Additional video-specific features beyond VISTA:
    - Optical flow magnitude/direction statistics for motion encoding
    - Frame-level appearance features: color histograms, HOG-like
      gradient descriptors from person bounding box regions
    - Temporal dynamics: velocity/acceleration of 2D keypoints

    Feature categories:
    1. 2D pose keypoints with torso normalization (VISTA core method)
    2. Keypoint velocity and acceleration over time
    3. Pairwise keypoint distances (hand-hand, hand-head, etc.)
    4. Optical flow motion features (magnitude + direction stats)
    5. Appearance features: color histograms from person region
    6. Covariance of joint trajectories (MSDFE-inspired)
    """

    # MediaPipe Pose landmark indices mapped to our 14-keypoint skeleton:
    # 0: head (nose)           -> mp landmark 0
    # 1: neck (midpoint of shoulders, computed)
    # 2: left_shoulder         -> mp landmark 11
    # 3: right_shoulder        -> mp landmark 12
    # 4: left_elbow            -> mp landmark 13
    # 5: right_elbow           -> mp landmark 14
    # 6: left_wrist            -> mp landmark 15
    # 7: right_wrist           -> mp landmark 16
    # 8: left_hip              -> mp landmark 23
    # 9: right_hip             -> mp landmark 24
    # 10: left_knee            -> mp landmark 25
    # 11: right_knee           -> mp landmark 26
    # 12: left_ankle           -> mp landmark 27
    # 13: right_ankle          -> mp landmark 28
    NUM_KEYPOINTS = 14

    # Mapping from our keypoint index to MediaPipe Pose landmark index
    # (neck=1 is computed as midpoint of shoulders, not directly mapped)
    MP_LANDMARK_MAP = {
        0: 0,    # nose -> head
        2: 11,   # left shoulder
        3: 12,   # right shoulder
        4: 13,   # left elbow
        5: 14,   # right elbow
        6: 15,   # left wrist
        7: 16,   # right wrist
        8: 23,   # left hip
        9: 24,   # right hip
        10: 25,  # left knee
        11: 26,  # right knee
        12: 27,  # left ankle
        13: 28,  # right ankle
    }

    # Keypoint pairs for distance features (same as original)
    KEYPOINT_PAIRS = [
        (6, 7),    # left wrist - right wrist
        (12, 13),  # left ankle - right ankle
        (6, 0),    # left wrist - head
        (7, 0),    # right wrist - head
        (6, 8),    # left wrist - left hip
        (7, 9),    # right wrist - right hip
        (2, 3),    # left shoulder - right shoulder
        (8, 9),    # left hip - right hip
    ]

    def __init__(self, config):
        self.n_sample_frames = config.VIDEO_SAMPLE_FRAMES
        self.resize = config.VIDEO_RESIZE

        # Initialize MediaPipe Pose
        BaseOptions = python.BaseOptions
        PoseLandmarker = vision.PoseLandmarker
        PoseLandmarkerOptions = vision.PoseLandmarkerOptions
        VisionRunningMode = vision.RunningMode

        self.pose = PoseLandmarker.create_from_options(
            PoseLandmarkerOptions(
                base_options=BaseOptions(
                    model_asset_path="pose_landmarker_full.task"
                ),
                running_mode=VisionRunningMode.IMAGE,
                num_poses=1,
                min_pose_detection_confidence=0.5,
                min_pose_presence_confidence=0.5,
                min_tracking_confidence=0.5,
            )
        )

    def __del__(self):
        """Clean up MediaPipe resources."""
        if hasattr(self, 'pose'):
            self.pose.close()

    def _read_video_frames(self, video_path, loss_rate=0.0, loss_rng=None):
        """Read and sample frames from a video file."""
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            logger.debug(f"Cannot open video: {video_path}")
            return None

        frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(frame)
        cap.release()

        if len(frames) == 0:
            return None

        # Uniformly sample frames
        n = len(frames)
        if n <= self.n_sample_frames:
            indices = list(range(n))
        else:
            indices = np.linspace(0, n - 1, self.n_sample_frames, dtype=int)

        sampled = [frames[i] for i in indices]

        # Resize for efficiency
        resized = []
        for f in sampled:
            r = cv2.resize(f, (self.resize[1], self.resize[0]))
            resized.append(r)


        # --- Apply RGB video frame loss BEFORE feature extraction ---
        if loss_rate > 0.0 and loss_rng is not None and len(resized) > 0:
            import math
            n_frames = len(resized)
            n_zero = max(1, math.ceil(n_frames * loss_rate))
            frames_to_zero = loss_rng.choice(n_frames, size=n_zero, replace=False)
            for fidx in frames_to_zero:
                resized[fidx] = np.zeros_like(resized[fidx])
            logger.debug(
                f"Video frame loss: zeroed {n_zero}/{n_frames} frames "
                f"({frames_to_zero.tolist()})"
            )

        return resized

    def _estimate_keypoints_mediapipe(self, frame):
        """
        Estimate 2D keypoints using MediaPipe Pose.

        Returns:
            keypoints: (14, 2) array of (x, y) normalized to [0, 1]
            bbox: (y_min, y_max, x_min, x_max) bounding box or None
            visibility: (14,) array of visibility scores
        """
        h, w = frame.shape[:2]

        # MediaPipe expects RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # results = self.pose.process(rgb_frame)
        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )
        results = self.pose.detect(mp_image)

        keypoints = np.zeros((self.NUM_KEYPOINTS, 2))
        visibility = np.zeros(self.NUM_KEYPOINTS)
        bbox = None

        # if results.pose_landmarks is not None:
        #    landmarks = results.pose_landmarks.landmark

        if results.pose_landmarks:
            landmarks = results.pose_landmarks[0]

            # Map MediaPipe landmarks to our 14-keypoint skeleton
            for our_idx, mp_idx in self.MP_LANDMARK_MAP.items():
                # lm = landmarks[mp_idx]
                # keypoints[our_idx] = [lm.x, lm.y]  # already normalized [0, 1]
                # visibility[our_idx] = lm.visibility

                lm = landmarks[mp_idx]
                keypoints[our_idx] = [lm.x, lm.y]
                visibility[our_idx] = lm.visibility if hasattr(lm, "visibility") else 1.0

            # Neck (index 1) = midpoint of left shoulder and right shoulder
            lm_ls = landmarks[11]
            lm_rs = landmarks[12]
            keypoints[1] = [
                (lm_ls.x + lm_rs.x) / 2.0,
                (lm_ls.y + lm_rs.y) / 2.0,
            ]
            visibility[1] = min(lm_ls.visibility, lm_rs.visibility)

            # Compute bounding box from all mapped landmarks
            all_x = [keypoints[i, 0] * w for i in range(self.NUM_KEYPOINTS)]
            all_y = [keypoints[i, 1] * h for i in range(self.NUM_KEYPOINTS)]
            margin = 0.05  # 5% margin
            x_min = max(0, int(min(all_x) - margin * w))
            x_max = min(w, int(max(all_x) + margin * w))
            y_min = max(0, int(min(all_y) - margin * h))
            y_max = min(h, int(max(all_y) + margin * h))

            if x_max > x_min and y_max > y_min:
                bbox = (y_min, y_max, x_min, x_max)

        return keypoints, bbox, visibility

    def _normalize_keypoints(self, keypoints_seq):
        """
        VISTA-style torso normalization for 2D keypoints.
        Translate to torso center, scale by torso size.

        Args:
            keypoints_seq: (frames, 14, 2)
        Returns:
            normalized (frames, 14, 2)
        """
        # Torso center = midpoint of (neck, left_hip, right_hip)
        torso_center = np.mean(keypoints_seq[:, [1, 8, 9], :], axis=1, keepdims=True)
        kp_norm = keypoints_seq - torso_center

        # Scale by shoulder-hip distance (torso size)
        shoulder_mid = np.mean(keypoints_seq[:, [2, 3], :], axis=1)
        hip_mid = np.mean(keypoints_seq[:, [8, 9], :], axis=1)
        torso_len = np.linalg.norm(shoulder_mid - hip_mid, axis=1, keepdims=True)
        torso_len = np.clip(torso_len, 1e-6, None)
        kp_norm = kp_norm / torso_len[:, np.newaxis, :]

        return kp_norm

    def _temporal_statistics(self, signal_2d):
        """
        Compute temporal statistics per feature column.
        Same stats as VISTA paper: mean, std, RMS, skewness, kurtosis,
        range, median, Q1, Q3.
        """
        features = []
        for col in range(signal_2d.shape[1]):
            s = signal_2d[:, col]
            features.extend([
                np.mean(s),
                np.std(s),
                np.sqrt(np.mean(s**2)),
                stats.skew(s) if len(s) > 2 else 0.0,
                stats.kurtosis(s) if len(s) > 3 else 0.0,
                np.max(s) - np.min(s),
                np.median(s),
                np.percentile(s, 25),
                np.percentile(s, 75),
            ])
        return np.array(features)

    def _compute_optical_flow(self, frames):
        """
        Compute dense optical flow between consecutive frames.
        Returns per-frame flow magnitude and direction statistics.
        """
        if len(frames) < 2:
            return np.zeros((1, 8))

        flow_features = []
        prev_gray = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)

        for i in range(1, len(frames)):
            curr_gray = cv2.cvtColor(frames[i], cv2.COLOR_BGR2GRAY)
            flow = cv2.calcOpticalFlowFarneback(
                prev_gray, curr_gray, None,
                pyr_scale=0.5, levels=3, winsize=15,
                iterations=3, poly_n=5, poly_sigma=1.2, flags=0
            )
            mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])

            flow_features.append([
                np.mean(mag),
                np.std(mag),
                np.max(mag),
                np.sum(mag > np.mean(mag)),  # active pixel count
                np.mean(ang),
                np.std(ang),
                # Circular histogram entropy of direction (4 quadrants)
                *[np.sum((ang >= q * np.pi / 2) & (ang < (q + 1) * np.pi / 2))
                  / (mag.size + 1e-12) for q in range(4)]
            ])
            prev_gray = curr_gray

        return np.array(flow_features)  # (n_frames-1, 10)

    def _color_histogram_features(self, frames, bbox_list):
        """
        Extract color histogram features from person region.
        Inspired by VISTA's use of visual appearance information.
        """
        hist_features = []
        for frame, bbox in zip(frames, bbox_list):
            if bbox is None:
                hist_features.append(np.zeros(48))
                continue
            y1, y2, x1, x2 = bbox
            roi = frame[y1:y2, x1:x2]
            if roi.size == 0:
                hist_features.append(np.zeros(48))
                continue

            hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
            # H: 16 bins, S: 16 bins, V: 16 bins
            h_hist = cv2.calcHist([hsv], [0], None, [16], [0, 180]).ravel()
            s_hist = cv2.calcHist([hsv], [1], None, [16], [0, 256]).ravel()
            v_hist = cv2.calcHist([hsv], [2], None, [16], [0, 256]).ravel()

            total = h_hist.sum() + 1e-12
            h_hist /= total
            s_hist /= total
            v_hist /= total

            hist_features.append(np.concatenate([h_hist, s_hist, v_hist]))

        return np.array(hist_features)  # (n_frames, 48)

    def extract(self, video_path, loss_rate=0.0, loss_rng=None):
        """
        Extract VISTA-inspired feature vector from an RGB video.

        Pipeline:
        1. Read & sample frames
        2. Estimate 2D keypoints per frame using MediaPipe Pose
        3. Normalize keypoints to torso (VISTA method)
        4. Compute temporal statistics on keypoint trajectories
        5. Compute optical flow motion features
        6. Compute color histogram appearance features
        7. Concatenate all features

        Args:
            video_path: path to .avi video file
        Returns:
            1D feature vector or None
        """
        frames = self._read_video_frames(video_path, loss_rate=loss_rate, loss_rng=loss_rng)
        if frames is None or len(frames) < 2:
            return None

        # --- Step 1: Estimate keypoints per frame using MediaPipe ---
        keypoints_seq = []
        bboxes = []
        visibility_seq = []

        for frame in frames:
            kp, bbox, vis = self._estimate_keypoints_mediapipe(frame)
            keypoints_seq.append(kp)
            visibility_seq.append(vis)

            if bbox is not None:
                bboxes.append(bbox)
            else:
                # Fallback bbox: central region of frame
                h, w = frame.shape[:2]
                bboxes.append((h // 4, 3 * h // 4, w // 4, 3 * w // 4))

        keypoints_seq = np.array(keypoints_seq)      # (n_frames, 14, 2)
        visibility_seq = np.array(visibility_seq)      # (n_frames, 14)

        # --- Step 1b: Interpolate low-visibility keypoints ---
        # If MediaPipe failed on some frames, interpolate from neighbors
        for kp_idx in range(self.NUM_KEYPOINTS):
            vis = visibility_seq[:, kp_idx]
            low_vis_mask = vis < 0.3
            if low_vis_mask.all():
                # No reliable detections for this keypoint; leave as-is
                continue
            if low_vis_mask.any() and not low_vis_mask.all():
                good_frames = np.where(~low_vis_mask)[0]
                bad_frames = np.where(low_vis_mask)[0]
                for axis in range(2):
                    good_vals = keypoints_seq[good_frames, kp_idx, axis]
                    interp_vals = np.interp(bad_frames, good_frames, good_vals)
                    keypoints_seq[bad_frames, kp_idx, axis] = interp_vals

        # --- Step 2: VISTA-style torso normalization ---
        kp_norm = self._normalize_keypoints(keypoints_seq)

        # --- Step 3: Keypoint position features (temporal statistics) ---
        kp_flat = kp_norm.reshape(kp_norm.shape[0], -1)  # (frames, 28)
        pos_feats = self._temporal_statistics(kp_flat)

        # --- Step 4: Keypoint velocity features ---
        if kp_flat.shape[0] > 1:
            velocity = np.diff(kp_flat, axis=0)
            velocity = np.vstack([velocity, velocity[-1:]])
        else:
            velocity = np.zeros_like(kp_flat)
        vel_feats = self._temporal_statistics(velocity)

        # --- Step 5: Keypoint acceleration features ---
        if velocity.shape[0] > 1:
            accel = np.diff(velocity, axis=0)
            accel = np.vstack([accel, accel[-1:]])
        else:
            accel = np.zeros_like(velocity)
        acc_feats = self._temporal_statistics(accel)

        # --- Step 6: Pairwise keypoint distance features ---
        distances = []
        for j1, j2 in self.KEYPOINT_PAIRS:
            dist = np.linalg.norm(kp_norm[:, j1, :] - kp_norm[:, j2, :], axis=1)
            distances.append(dist)
        dist_arr = np.array(distances).T  # (frames, n_pairs)
        dist_feats = self._temporal_statistics(dist_arr)

        # --- Step 7: Optical flow features ---
        flow_data = self._compute_optical_flow(frames)
        if flow_data.shape[0] > 0:
            flow_feats = self._temporal_statistics(flow_data)
        else:
            flow_feats = np.zeros(10 * 9)

        # --- Step 8: Color histogram features ---
        color_data = self._color_histogram_features(frames, bboxes)
        # Aggregate across frames
        color_agg = []
        for col in range(color_data.shape[1]):
            col_vals = color_data[:, col]
            color_agg.extend([np.mean(col_vals), np.std(col_vals)])
        color_feats = np.array(color_agg)

        # --- Step 9: Covariance of joint trajectories (MSDFE-inspired) ---
        if kp_flat.shape[0] > 1:
            cov_mat = np.cov(kp_flat.T)
            cov_feats = cov_mat[np.triu_indices(cov_mat.shape[0])]
        else:
            n = kp_flat.shape[1]
            cov_feats = np.zeros(n * (n + 1) // 2)

        # --- Concatenate all video features ---
        all_feats = np.concatenate([
            pos_feats,      # Keypoint position statistics
            vel_feats,      # Keypoint velocity statistics
            acc_feats,      # Keypoint acceleration statistics
            dist_feats,     # Pairwise distance statistics
            flow_feats,     # Optical flow statistics
            color_feats,    # Color histogram features
            cov_feats,      # Joint trajectory covariance
        ])

        return np.nan_to_num(all_feats, nan=0.0, posinf=0.0, neginf=0.0)

In [9]:
# =============================================================================
# SECTION 6: MULTIMODAL FUSION & CLASSIFICATION PIPELINE
# =============================================================================
#
# Memory-efficient streaming redesign
# ------------------------------------
#
#   Old flow (memory-intensive):
#       load_all_data()        → entire dataset in RAM (depth alone ~40 GB)
#       apply_raw_data_loss()  → still all in RAM
#       extract_features()     → still all in RAM
#
#   New flow (streaming):
#       build_sample_registry()          → filenames only (~KB total)
#       extract_features_streaming()
#           for each sample (one at a time):
#               load raw arrays for this sample
#               apply loss in-place
#               extract features
#               delete raw arrays + gc.collect() periodically
#               append compact feature vector to accumulator
#
# Peak memory ≈ one sample's raw data  +  all accumulated feature vectors.
# Feature vectors are O(KB) each, so the full features_dict fits in RAM.
# =============================================================================

class MultimodalHARPipeline:
    """Complete pipeline: discover files, extract features, fuse, classify."""

    def __init__(self, config):
        self.config          = config
        self.loader          = UTDMHADLoader(config)
        self.skel_extractor  = SkeletonFeatureExtractor()
        self.iner_extractor  = InertialFeatureExtractor(config)
        self.depth_extractor = DepthFeatureExtractor(config)
        self.video_extractor = VideoFeatureExtractor(config)
        self.scaler          = StandardScaler()
        # One RNG per modality – seeded from config.LOSS_SEED so results
        # are reproducible, but each advances independently.
        seed = config.LOSS_SEED
        self._skel_loss_rng  = np.random.RandomState(seed)
        self._iner_loss_rng  = np.random.RandomState(seed + 1)
        self._depth_loss_rng = np.random.RandomState(seed + 2)
        self._video_loss_rng = np.random.RandomState(seed + 3)

    # ------------------------------------------------------------------
    # Streaming feature extraction
    # ------------------------------------------------------------------

    def extract_features_streaming(self, registry):
        """
        Stream through the registry one sample at a time.

        For each sample:
          1. Load raw arrays from disk (this sample only)
          2. Apply modality-specific loss in-place
          3. Extract features
          4. Delete raw arrays immediately
          5. Store only the compact feature vector

        Args:
            registry : dict from UTDMHADLoader.build_sample_registry()

        Returns:
            features_dict : {key: {features, label, subject,
                                   modalities, modality_sizes,
                                   modality_features}}
        """
        features_dict = {}
        total = len(registry)

        for i, (key, meta) in enumerate(sorted(registry.items())):
            if i == 0 or (i + 1) % 50 == 0:
                logger.info(f"Extracting features: {i+1}/{total}  [{key}]")

            feature_parts = []

            # ── Skeleton ──────────────────────────────────────────────
            if self.config.USE_SKELETON and meta['skeleton_path']:
                skel = UTDMHADLoader.load_skeleton_from_file(meta['skeleton_path'])
                if skel is not None:
                    apply_skeleton_loss(skel, self.config.LOSS_RATE, self._skel_loss_rng)
                    feats = self.skel_extractor.extract(skel)
                    del skel
                    if feats is not None:
                        feature_parts.append(('skeleton', feats))

            # ── Inertial ───────────────────────────────────────────────
            if self.config.USE_INERTIAL and meta['inertial_path']:
                iner = UTDMHADLoader.load_inertial_from_file(meta['inertial_path'])
                if iner is not None:
                    apply_inertial_loss(iner, self.config.LOSS_RATE, self._iner_loss_rng)
                    feats = self.iner_extractor.extract(iner)
                    del iner
                    if feats is not None:
                        feature_parts.append(('inertial', feats))

            # ── Depth ──────────────────────────────────────────────────
            if self.config.USE_DEPTH and meta['depth_path']:
                depth = UTDMHADLoader.load_depth_from_file(meta['depth_path'])
                if depth is not None:
                    apply_depth_loss(depth, self.config.LOSS_RATE, self._depth_loss_rng)
                    feats = self.depth_extractor.extract(depth)
                    del depth
                    if feats is not None:
                        feature_parts.append(('depth', feats))

            # ── Video ──────────────────────────────────────────────────
            # VideoFeatureExtractor reads and discards frames internally,
            # so no explicit del is needed here.
            if self.config.USE_VIDEO and meta['video_path']:
                feats = self.video_extractor.extract(
                    meta['video_path'],
                    loss_rate=self.config.LOSS_RATE,
                    loss_rng=self._video_loss_rng,
                )
                if feats is not None:
                    feature_parts.append(('video', feats))

            # ── Accumulate feature vector ─────────────────────────────
            if feature_parts:
                concatenated = np.concatenate([f for _, f in feature_parts])
                features_dict[key] = {
                    'features':          concatenated,
                    'label':             meta['action'],
                    'subject':           meta['subject'],
                    'modalities':        [n for n, _ in feature_parts],
                    'modality_sizes':    {n: len(f) for n, f in feature_parts},
                    'modality_features': {n: f for n, f in feature_parts},
                }

            # Periodic GC to release any memory held by the OS
            if (i + 1) % 20 == 0:
                gc.collect()

        logger.info(f"Extraction complete – {len(features_dict)}/{total} samples")
        if features_dict:
            ex = next(iter(features_dict.values()))
            logger.info(f"  Total feature dim: {len(ex['features'])}")
            for mod, sz in ex['modality_sizes'].items():
                logger.info(f"    {mod:12s}: {sz} features")

        return features_dict

    # ------------------------------------------------------------------
    # Save features
    # ------------------------------------------------------------------

    def save_features(self, features_dict):
        """
        Save extracted features to disk for offline use.

        Output files:
          X_feat.pkl       – list of per-modality feature dicts
          y.npy            – encoded action labels
          subjects.npy     – subject IDs
          label_encoder.pkl
        """
        out_dir = self.config.FEATURES_DIR
        os.makedirs(out_dir, exist_ok=True)
        print("Out directory:", out_dir)

        keys = sorted(features_dict.keys())
        X_feat, y_raw, subjects = [], [], []

        for k in keys:
            entry    = features_dict[k]
            mod_feat = entry.get('modality_features', {})
            X_feat.append({
                'skeleton_feat': mod_feat.get('skeleton', np.array([])),
                'sensor_feat':   mod_feat.get('inertial', np.array([])),
                'depth_feat':    mod_feat.get('depth',    np.array([])),
                'video_feat':    mod_feat.get('video',    np.array([])),
            })
            y_raw.append(entry['label'])
            subjects.append(entry['subject'])

        y_raw    = np.array(y_raw)
        subjects = np.array(subjects)
        le = LabelEncoder()
        y  = le.fit_transform(y_raw)

        joblib.dump(X_feat, os.path.join(out_dir, 'X_feat.pkl'))
        np.save(os.path.join(out_dir, 'y.npy'),        y)
        np.save(os.path.join(out_dir, 'subjects.npy'), subjects)
        joblib.dump(le,     os.path.join(out_dir, 'label_encoder.pkl'))

        logger.info(f"Features saved to '{out_dir}/':")
        logger.info(f"  X_feat.pkl : {len(X_feat)} samples")
        first = X_feat[0]
        for mk in ['skeleton_feat', 'sensor_feat', 'depth_feat', 'video_feat']:
            dim = first[mk].shape[0] if first[mk].size > 0 else 0
            logger.info(f"    {mk:18s}: {dim} dims")
        total_dim = sum(first[m].shape[0] for m in first if first[m].size > 0)
        logger.info(f"    {'TOTAL':18s}: {total_dim} dims")
        logger.info(f"  y.npy      : {y.shape}  ({len(le.classes_)} classes)")
        logger.info(f"  subjects   : {sorted(np.unique(subjects).tolist())}")

    # ------------------------------------------------------------------
    # Train / test split
    # ------------------------------------------------------------------

    def split_train_test(self, features_dict):
        """Subject-based train/test split (odd subjects = train)."""
        X_train, y_train, X_test, y_test = [], [], [], []
        for key, entry in features_dict.items():
            if entry['subject'] in self.config.TRAIN_SUBJECTS:
                X_train.append(entry['features']); y_train.append(entry['label'])
            elif entry['subject'] in self.config.TEST_SUBJECTS:
                X_test.append(entry['features']);  y_test.append(entry['label'])

        X_train = np.array(X_train); y_train = np.array(y_train)
        X_test  = np.array(X_test);  y_test  = np.array(y_test)
        logger.info(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]} | "
                    f"Dim: {X_train.shape[1]} | Classes: {len(np.unique(y_train))}")
        return X_train, y_train, X_test, y_test

    # ------------------------------------------------------------------
    # Train and evaluate
    # ------------------------------------------------------------------

    def train_and_evaluate(self, X_train, y_train, X_test, y_test):
        """Standardise, train Random Forest, evaluate."""
        X_tr = np.nan_to_num(self.scaler.fit_transform(X_train), nan=0., posinf=0., neginf=0.)
        X_te = np.nan_to_num(self.scaler.transform(X_test),      nan=0., posinf=0., neginf=0.)

        rf = RandomForestClassifier(
            n_estimators      = self.config.RF_N_ESTIMATORS,
            max_depth         = self.config.RF_MAX_DEPTH,
            min_samples_split = self.config.RF_MIN_SAMPLES_SPLIT,
            min_samples_leaf  = self.config.RF_MIN_SAMPLES_LEAF,
            random_state      = self.config.RF_RANDOM_STATE,
            n_jobs            = self.config.RF_N_JOBS,
            class_weight      = 'balanced',
        )

        logger.info("Training Random Forest ...")
        t0 = time.time()
        rf.fit(X_tr, y_train)
        train_time = time.time() - t0
        logger.info(f"Training done in {train_time:.2f}s")

        y_pred = rf.predict(X_te)
        acc    = accuracy_score(y_test, y_pred)
        f1_mac = f1_score(y_test, y_pred, average='macro')
        f1_wtd = f1_score(y_test, y_pred, average='weighted')

        logger.info(f"\n{'='*60}")
        logger.info("RESULTS")
        logger.info(f"{'='*60}")
        logger.info(f"Accuracy        : {acc*100:.2f}%")
        logger.info(f"Macro F1        : {f1_mac*100:.2f}%")
        logger.info(f"Weighted F1     : {f1_wtd*100:.2f}%")

        action_names  = [f"Action {i}" for i in range(1, self.config.NUM_ACTIONS + 1)]
        unique_labels = sorted(np.unique(np.concatenate([y_test, y_pred])))
        target_names  = [action_names[l - 1] for l in unique_labels]
        logger.info("\nClassification Report:\n" +
                    classification_report(y_test, y_pred, labels=unique_labels,
                                          target_names=target_names, digits=3))

        logger.info("5-fold CV on training set ...")
        cv = cross_val_score(rf, X_tr, y_train, cv=5, scoring='accuracy')
        logger.info(f"CV Accuracy: {cv.mean()*100:.2f}% +/- {cv.std()*100:.2f}%")

        top_k = 20
        top_idx = np.argsort(rf.feature_importances_)[-top_k:][::-1]
        logger.info(f"\nTop {top_k} feature importances:")
        for idx in top_idx:
            logger.info(f"  Feature {idx}: {rf.feature_importances_[idx]:.4f}")

        return {
            'accuracy':    acc,
            'f1_macro':    f1_mac,
            'f1_weighted': f1_wtd,
            'cv_mean':     cv.mean(),
            'cv_std':      cv.std(),
            'train_time':  train_time,
            'model':       rf,
            'y_pred':      y_pred,
        }

    # ------------------------------------------------------------------
    # Main entry point
    # ------------------------------------------------------------------

    def run(self):
        """Execute the full pipeline."""
        logger.info("=" * 60)
        logger.info("MULTIMODAL HAR PIPELINE – UTD-MHAD  (streaming / lazy-load)")
        logger.info("=" * 60)
        logger.info(f"Modalities : skeleton={self.config.USE_SKELETON}  "
                    f"inertial={self.config.USE_INERTIAL}  "
                    f"depth={self.config.USE_DEPTH}  "
                    f"video={self.config.USE_VIDEO}")
        logger.info(f"Loss rate  : {self.config.LOSS_RATE*100:.0f}%")
        logger.info(f"Train subs : {self.config.TRAIN_SUBJECTS}")
        logger.info(f"Test subs  : {self.config.TEST_SUBJECTS}")

        # Step 1 – build filename-only registry  (no raw data loaded)
        logger.info("\n--- Step 1: Building sample registry (filenames only) ---")
        registry = self.loader.build_sample_registry()
        if not registry:
            logger.error("No files found.  Check DATASET_ROOT in Config.")
            logger.info(f"  Expected: {self.config.DATASET_ROOT}")
            logger.info("  Download: https://personal.utdallas.edu/~kehtar/UTD-MHAD.html")
            return None

        # Step 2 – stream: load one sample → apply loss → extract → discard
        logger.info("\n--- Step 2: Streaming feature extraction ---")
        features_dict = self.extract_features_streaming(registry)
        if not features_dict:
            logger.error("No features extracted!")
            return None

        # Step 2b – persist to disk
        logger.info("\n--- Step 2b: Saving features ---")
        self.save_features(features_dict)

        # Step 3 – split
        logger.info("\n--- Step 3: Train/test split ---")
        X_train, y_train, X_test, y_test = self.split_train_test(features_dict)
        if X_train.shape[0] == 0 or X_test.shape[0] == 0:
            logger.error("Empty train or test set!")
            return None

        # Step 4 – train & evaluate
        logger.info("\n--- Step 4: Training and evaluation ---")
        return self.train_and_evaluate(X_train, y_train, X_test, y_test)


In [10]:
# =============================================================================
# SECTION 7: DEMO MODE (generates synthetic data for testing)
# =============================================================================

def create_synthetic_dataset(root_dir, n_actions=27, n_subjects=8, n_trials=4):
    """
    Create a synthetic UTD-MHAD-like dataset for testing the pipeline.
    This generates random data with the correct structure and file naming.
    """
    logger.info("Creating synthetic dataset for demonstration...")

    for modality, subdir in [("skeleton", "Skeleton"), ("inertial", "Inertial"),
                              ("depth", "Depth")]:
        mod_dir = os.path.join(root_dir, subdir)
        os.makedirs(mod_dir, exist_ok=True)

        for a in range(1, n_actions + 1):
            for s in range(1, n_subjects + 1):
                for t in range(1, n_trials + 1):
                    fname = f"a{a}_s{s}_t{t}_{modality}.mat"
                    fpath = os.path.join(mod_dir, fname)

                    if os.path.exists(fpath):
                        continue

                    n_frames = np.random.randint(30, 120)

                    if modality == "skeleton":
                        # (frames, 20, 3) with class-dependent patterns
                        base = np.random.randn(n_frames, 20, 3) * 0.1
                        # Add action-specific signal
                        base[:, :, 0] += np.sin(np.linspace(0, a * np.pi, n_frames))[:, None]
                        base[:, :, 1] += np.cos(np.linspace(0, a * 0.5 * np.pi, n_frames))[:, None]
                        from scipy.io import savemat
                        savemat(fpath, {'d_skel': base.reshape(n_frames, 60)})

                    elif modality == "inertial":
                        # (samples, 6) with class-dependent patterns
                        n_samples = np.random.randint(100, 500)
                        data = np.random.randn(n_samples, 6) * 0.5
                        # Action-specific frequency
                        freq = a * 0.5
                        t_axis = np.linspace(0, 2 * np.pi * freq, n_samples)
                        data[:, 0] += np.sin(t_axis)
                        data[:, 3] += np.cos(t_axis * 0.7)
                        from scipy.io import savemat
                        savemat(fpath, {'d_iner': data})

                    elif modality == "depth":
                        # (height, width, frames) - small for speed
                        h, w = 60, 80
                        n_depth_frames = min(n_frames, 20)
                        depth = np.zeros((h, w, n_depth_frames))
                        for fr in range(n_depth_frames):
                            # Create a blob that moves based on action class
                            cy = h // 2 + int(5 * np.sin(a * fr / 10.0))
                            cx = w // 2 + int(5 * np.cos(a * fr / 10.0))
                            Y, X = np.ogrid[:h, :w]
                            mask = (Y - cy)**2 + (X - cx)**2 < (10 + a)**2
                            depth[:, :, fr][mask] = 1000 + a * 50 + np.random.rand() * 100
                        from scipy.io import savemat
                        savemat(fpath, {'d_depth': depth})

    logger.info(f"Synthetic dataset created at {root_dir}")


# =============================================================================
# SECTION 8: MAIN EXECUTION
# =============================================================================

def main():
    """Main entry point."""

    config = Config()

    # Check if real dataset exists
    if not os.path.isdir(config.DATASET_ROOT):
        logger.info(f"Dataset not found at '{config.DATASET_ROOT}'")
        logger.info("Creating synthetic dataset for demonstration...")
        os.makedirs(config.DATASET_ROOT, exist_ok=True)
        create_synthetic_dataset(config.DATASET_ROOT)

    # Run pipeline
    pipeline = MultimodalHARPipeline(config)
    results = pipeline.run()

    if results:
        logger.info(f"\n{'='*60}")
        logger.info("FINAL SUMMARY")
        logger.info(f"{'='*60}")
        logger.info(f"Accuracy:         {results['accuracy']*100:.2f}%")
        logger.info(f"Macro F1:         {results['f1_macro']*100:.2f}%")
        logger.info(f"Weighted F1:      {results['f1_weighted']*100:.2f}%")
        logger.info(f"CV Accuracy:      {results['cv_mean']*100:.2f}% "
                     f"(+/- {results['cv_std']*100:.2f}%)")
        logger.info(f"Training Time:    {results['train_time']:.2f}s")


if __name__ == "__main__":
    main()


2026-05-10 19:22:08,088 - INFO - ============================================================
2026-05-10 19:22:08,089 - INFO - MULTIMODAL HAR PIPELINE – UTD-MHAD  (streaming / lazy-load)
2026-05-10 19:22:08,089 - INFO - ============================================================
2026-05-10 19:22:08,090 - INFO - Modalities : skeleton=True  inertial=True  depth=True  video=True
2026-05-10 19:22:08,090 - INFO - Loss rate  : 50%
2026-05-10 19:22:08,090 - INFO - Train subs : [1, 3, 5, 7]
2026-05-10 19:22:08,091 - INFO - Test subs  : [2, 4, 6, 8]
2026-05-10 19:22:08,092 - INFO - 
--- Step 1: Building sample registry (filenames only) ---
2026-05-10 19:22:08,109 - INFO - Registry built – 861 unique samples | skeleton:861  inertial:861  depth:861  video:861
2026-05-10 19:22:08,110 - INFO - 
--- Step 2: Streaming feature extraction ---
2026-05-10 19:22:08,111 - INFO - Extracting features: 1/861  [a10_s1_t1]
2026-05-10 19:22:51,312 - INFO - Extracting features: 50/861  [a11_s5_t2]
2026-05-10 19:

Out directory: features_loss_50


2026-05-10 19:34:34,088 - INFO - Features saved to 'features_loss_50/':
2026-05-10 19:34:34,090 - INFO -   X_feat.pkl : 861 samples
2026-05-10 19:34:34,090 - INFO -     skeleton_feat     : 1879 dims
2026-05-10 19:34:34,091 - INFO -     sensor_feat       : 652 dims
2026-05-10 19:34:34,092 - INFO -     depth_feat        : 5508 dims
2026-05-10 19:34:34,092 - INFO -     video_feat        : 1420 dims
2026-05-10 19:34:34,093 - INFO -     TOTAL             : 9459 dims
2026-05-10 19:34:34,093 - INFO -   y.npy      : (861,)  (27 classes)
2026-05-10 19:34:34,094 - INFO -   subjects   : [1, 2, 3, 4, 5, 6, 7, 8]
2026-05-10 19:34:34,095 - INFO - 
--- Step 3: Train/test split ---
2026-05-10 19:34:34,126 - INFO - Train: 431 | Test: 430 | Dim: 9459 | Classes: 27
2026-05-10 19:34:34,128 - INFO - 
--- Step 4: Training and evaluation ---
2026-05-10 19:34:34,265 - INFO - Training Random Forest ...
2026-05-10 19:34:34,915 - INFO - Training done in 0.65s
2026-05-10 19:34:35,079 - INFO - 
2026-05-10 19:34:35